In [2]:
import argparse
import sys
from pathlib import Path
 
import pandas as pd
import numpy as np

In [3]:
PHQ4_ITEMS   = ["phq4-1", "phq4-2", "phq4-3", "phq4-4"]
SSE3_ITEMS   = ["sse3-1", "sse3-2", "sse3-3", "sse3-4"]
SCORE_COLS   = ["phq4_score", "pam", "stress", "social_level"]
TIMING_COLS  = ["phq4_resp_mean", "phq4_resp_median",
                "sse3_resp_mean",  "sse3_resp_median",
                "avg_ema_spent_time"]
ALL_EMA_COLS = PHQ4_ITEMS + SSE3_ITEMS + SCORE_COLS + TIMING_COLS

In [4]:
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Parse the YYYYMMDD date column into proper dates
    df["date"] = pd.to_datetime(df["day"].astype(str), format="%Y%m%d")
    # ISO week label  →  "YYYY-Www"  (consistent across year boundaries)
    df["year_week"] = df["date"].dt.isocalendar().year.astype(str) + "-W" + \
                      df["date"].dt.isocalendar().week.astype(str).str.zfill(2)
    return df

In [5]:
# Whole table analysis of temporal coverage, survey counts, and missing data patterns

def temporal_coverage(df: pd.DataFrame) -> dict:
    """Global and per-student date range."""
    global_min = df["date"].min()
    global_max = df["date"].max()
    total_days = (global_max - global_min).days + 1
    total_weeks = total_days / 7
 
    per_student = (
        df.groupby("uid")["date"]
        .agg(start="min", end="max")
        .assign(span_days=lambda x: (x["end"] - x["start"]).dt.days + 1)
    )
 
    return {
        "global_start":       global_min.date(),
        "global_end":         global_max.date(),
        "total_calendar_days": total_days,
        "total_calendar_weeks": round(total_weeks, 2),
        "n_students":         df["uid"].nunique(),
        "per_student":        per_student,
    }
 
def weekly_survey_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    For every (student, ISO-week) pair count the number of survey responses.
    A 'response' = a row that is not entirely NaN across EMA columns.
    """
    ema_present = ALL_EMA_COLS  # columns that actually exist in the file
    existing    = [c for c in ema_present if c in df.columns]
 
    df = df.copy()
    df["has_response"] = df[existing].notna().any(axis=1)
 
    weekly = (
        df[df["has_response"]]
        .groupby(["uid", "year_week"])
        .size()
        .reset_index(name="surveys_in_week")
    )
    return weekly

def per_student_survey_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Total number of survey responses submitted per student."""
    existing = [c for c in ALL_EMA_COLS if c in df.columns]

    counts = (
        df[df[existing].notna().any(axis=1)]
        .groupby("uid")
        .size()
        .reset_index(name="total_surveys")
        .sort_values("total_surveys", ascending=False)
        .reset_index(drop=True)
    )
    return counts
 
def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    """Missing-value counts and percentages for every EMA column."""
    existing = [c for c in ALL_EMA_COLS if c in df.columns]
    total    = len(df)
 
    report = pd.DataFrame({
        "column":    existing,
        "missing_n": [df[c].isna().sum() for c in existing],
        "missing_%": [round(df[c].isna().mean() * 100, 2) for c in existing],
    })
    return report.sort_values("missing_%", ascending=False).reset_index(drop=True)
 
def per_student_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each student: total rows, rows with ANY EMA data, fully-missing rows.
    Useful for spotting students who dropped out early or never responded.
    """
    existing = [c for c in ALL_EMA_COLS if c in df.columns]
    result = []
    for uid, grp in df.groupby("uid"):
        total      = len(grp)
        any_data   = grp[existing].notna().any(axis=1).sum()
        all_missing = total - any_data
        result.append({
            "uid":            uid,
            "total_rows":     total,
            "rows_with_data": any_data,
            "fully_missing":  all_missing,
            "pct_missing":    round(all_missing / total * 100, 2),
        })
    return pd.DataFrame(result).sort_values("pct_missing", ascending=False).reset_index(drop=True)
 
 
def print_section(title: str):
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print(f"{'═' * 60}")

In [6]:
def run_whole_table_analysis(path: str, save: bool = False):
    print(f"\n[EDA] Loading: {path}")
    df = load_data(path)
    print(f"      Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"      Students: {df['uid'].nunique()}")
 
    lines = []  # collects text for optional markdown report

    # ── 1. Temporal coverage ──────────────────────────────────────────────────
    print_section("1. Temporal Coverage")
    cov = temporal_coverage(df)
 
    print(f"  Global start (earliest survey)  : {cov['global_start']}")
    print(f"  Global end   (latest survey)    : {cov['global_end']}")
    print(f"  Total calendar days             : {cov['total_calendar_days']}")
    print(f"  Total calendar weeks            : {cov['total_calendar_weeks']}")
    print(f"  Number of students              : {cov['n_students']}")
 
    per = cov["per_student"]
    print(f"\n  Per-student span (days) — summary:")
    print(per["span_days"].describe().rename("span_days").to_string())
 
    print(f"\n  Earliest start dates (top 5 students):")
    print(per.nsmallest(5, "start")[["start","end","span_days"]].to_string())
 
    print(f"\n  Latest end dates (top 5 students):")
    print(per.nlargest(5, "end")[["start","end","span_days"]].to_string())
 
    lines += [
        "## 1. Temporal Coverage\n",
        f"- **Global start**: {cov['global_start']}",
        f"- **Global end**: {cov['global_end']}",
        f"- **Calendar days**: {cov['total_calendar_days']}",
        f"- **Students**: {cov['n_students']}\n",
        "### Per-student span (days)\n",
        per["span_days"].describe().to_frame().to_markdown(), "\n",
    ]
 
    # ── 2. Weekly survey counts ───────────────────────────────────────────────
    print_section("2. Weekly Survey Count Analysis")
    weekly = weekly_survey_counts(df)
 
    max_surveys_per_week = weekly["surveys_in_week"].max()
    mean_surveys         = weekly["surveys_in_week"].mean()
    dist                 = weekly["surveys_in_week"].value_counts().sort_index()
 
    print(f"  Max surveys a student received in a single week : {max_surveys_per_week}")
    print(f"  Mean surveys per (student, week)                : {mean_surveys:.2f}")
    print(f"\n  Distribution of surveys-per-week across all (student, week) pairs:")
    print(dist.rename_axis("surveys_in_week").reset_index(name="count").to_string(index=False))
 
    # Who had the most survey-dense week?
    busiest = weekly.loc[weekly["surveys_in_week"].idxmax()]
    print(f"\n  Most survey-dense (student, week): uid={busiest['uid']}, "
          f"week={busiest['year_week']}, surveys={busiest['surveys_in_week']}")
 
    lines += [
        "\n## 2. Weekly Survey Counts\n",
        f"- **Max surveys in one week**: {max_surveys_per_week}",
        f"- **Mean surveys per (student, week)**: {mean_surveys:.2f}\n",
        "### Distribution of surveys per week\n",
        dist.rename_axis("surveys_in_week").reset_index(name="count").to_markdown(index=False), "\n",
    ]

    # ── 3. Per-student survey counts ───────────────────────────────────────────────
    print_section("3. Total Surveys per Student")
    sc = per_student_survey_counts(df)
    print(sc.to_string(index=False))
    print(f"\n  Mean : {sc['total_surveys'].mean():.1f}")
    print(f"  Min  : {sc['total_surveys'].min()}")
    print(f"  Max  : {sc['total_surveys'].max()}")

    lines += [
        "\n## 3. Total Surveys per Student\n",
        sc.to_markdown(index=False), "\n",
        f"\n**Mean surveys per student**: {sc['total_surveys'].mean():.1f}\n",
    ]
 
    # ── 4. Missing-value report ───────────────────────────────────────────────
    print_section("4. Missing-Value Report (EMA columns)")
    mv = missing_value_report(df)
    print(mv.to_string(index=False))
 
    lines += [
        "\n## 4. Missing Values per EMA Column\n",
        mv.to_markdown(index=False), "\n",
    ]
 
    # ── 5. Per-student missing ────────────────────────────────────────────────
    print_section("5. Per-Student Missing-Value Summary")
    ps = per_student_missing(df)
    print(ps.to_string(index=False))
 
    # Students with >50% missing
    heavy = ps[ps["pct_missing"] > 50]
    print(f"\n  Students with >50 % fully-missing rows: {len(heavy)}")
    if not heavy.empty:
        print(heavy.to_string(index=False))
 
    lines += [
        "\n## 5. Per-Student Missing Values\n",
        ps.to_markdown(index=False), "\n",
        f"\n**Students with >50% missing rows**: {len(heavy)}\n",
    ]
 
    # ── 6. Save report ────────────────────────────────────────────────────────
    if save:
        out = Path("../outputs/reports/eda_general_ema_report.md")
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text("\n".join(lines), encoding="utf-8")
        print(f"\n[EDA] Report saved → {out}")
        print("\n[EDA] Done.\n")

In [7]:
path = "../data/raw/college_experience_dataset/EMA/general_ema.csv"
run_whole_table_analysis(path, save=True)


[EDA] Loading: ../data/raw/college_experience_dataset/EMA/general_ema.csv
      Shape   : 217,155 rows × 21 columns
      Students: 220

════════════════════════════════════════════════════════════
  1. Temporal Coverage
════════════════════════════════════════════════════════════
  Global start (earliest survey)  : 2017-09-07
  Global end   (latest survey)    : 2022-07-04
  Total calendar days             : 1762
  Total calendar weeks            : 251.71
  Number of students              : 220

  Per-student span (days) — summary:
count     220.000000
mean     1093.372727
std       386.176548
min         2.000000
25%       929.000000
50%      1314.500000
75%      1350.250000
max      1647.000000

  Earliest start dates (top 5 students):
                                      start        end  span_days
uid                                                              
1ff6d7f34acb354430e7323a35ff7703 2017-09-07 2021-06-21       1384
3bb377ba0acb7d8916010184df36aa57 2017-09-07 2021-06-1

In [ ]:
# 